<a href="https://colab.research.google.com/github/LIONPANJSHIR/Row-Health/blob/main/Marketing_Channel_Performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Phase 3 : Marketing Channel Performance



Les performances de Social Media, Email, SEO et TV reflètent-elles une efficacité globale des canaux, ou dépendent-elles principalement de quelques campagnes exceptionnelles ?





In [1]:
# @title
import matplotlib.pyplot as plt
import seaborn as sns

# @title
from os import set_inheritable
from pathlib import Path
from google.colab import drive
import pandas as pd
import numpy as np

# drive.mount('/content/drive')
project_dir = Path("/content/drive/MyDrive/project/data_analytics/row_heatheart")


# mettre peoject_dir comme repertoire principale
%cd {project_dir}

# importer les données nettoyer

# claims.to_csv(clean_dir / "claims.csv", index=False)
# customers.to_csv(clean_dir / "customers.csv", index=False)
# campagne.to_csv(clean_dir / "campaigns.csv", index=False)

claims        = pd.read_csv(project_dir / "data/clean/claims.csv")
customers     = pd.read_csv(project_dir / "data/clean/customers.csv")
campagne      = pd.read_csv(project_dir / "data/clean/campaigns.csv")
category_kpis = pd.read_csv(project_dir/'data/output/category_kpis')
type_kpis     = pd.read_csv(project_dir/'data/output/type_kpis')
platform_kpis = pd.read_csv(project_dir/'data/output/platform_kpis')
campaign_kpis = pd.read_csv(project_dir/'data/output/campaign_kpis')
table_1       = pd.read_csv(project_dir/'data/output/table_1')

/content/drive/MyDrive/project/data_analytics/row_heatheart


### .Q1) Le canal Social Media reste-t-il le plus efficace si l'on exclut ses campagnes dominantes ?

In [2]:
# @title

# Performance globale par plateforme
global_performance = (
    campaign_kpis
    .groupby("platform")
    .agg(
        nb_campagnes=("campaign_id", "nunique"),
        budget=("cost", "sum"),
        inscriptions=("inscriptions", "sum")
    )
)

global_performance["CPS"] = (
    global_performance["budget"]
    / global_performance["inscriptions"].replace(0, np.nan)
)

# Analyse de sensibilité : sans les 5 meilleures campagnes
top5_ids = campaign_kpis.nlargest(
    5, "inscriptions"
)["campaign_id"]

without_top5 = (
    campaign_kpis
    .loc[~campaign_kpis["campaign_id"].isin(top5_ids)]
    .groupby("platform")
    .agg(
        nb_campagnes=("campaign_id", "nunique"),
        budget=("cost", "sum"),
        inscriptions=("inscriptions", "sum")
    )
)

without_top5["CPS"] = (
    without_top5["budget"]
    / without_top5["inscriptions"].replace(0, np.nan)
)

# Comparaison
comparison = global_performance[
    ["inscriptions", "CPS"]
].join(
    without_top5[["inscriptions", "CPS"]],
    rsuffix="_hors_top5"
)

display(comparison.round(2))


,inscriptions,CPS,inscriptions_hors_top5,CPS_hors_top5
platform,,,,
Email,4130,4.04,851,18.21
SEO,4055,5.24,1236,15.08
Social Media,7610,2.25,1333,11.19
TV,494,10.48,494,10.48


### Insight 1 — Les performances des canaux marketing reposent fortement sur quelques campagnes dominantes

>L'analyse de sensibilité révèle une forte dépendance des performances d'acquisition à un nombre limité de campagnes.
>
>Après exclusion des cinq campagnes générant le plus d'inscriptions attribuées, **Social Media perd 82,5 % de ses inscriptions**, tandis que son coût par inscription passe de **2,25 \$ à 11,19 \$**, soit une multiplication par près de cinq.
>
>Cette tendance s'observe également sur Email et SEO, dont les coûts par inscription passent respectivement de 4,04 \$ à 18,21 \$ et de 5,24 \$ à 15,08 \$.
>
>**Interprétation business :** l'efficacité historique des plateformes ne reflète pas nécessairement une performance homogène de leurs campagnes. Les résultats agrégés peuvent masquer une forte dépendance à quelques initiatives particulièrement performantes.
>
>**Implication stratégique :** les décisions d'allocation budgétaire ne devraient pas reposer uniquement sur les KPI globaux des plateformes. Une analyse au niveau des campagnes et de leurs caractéristiques est nécessaire afin d'identifier les stratégies potentiellement reproductibles, tout en vérifiant la fiabilité des données d'attribution.

### .Q2) Une même catégorie de campagne présente-t-elle des performances différentes selon sa plateforme de diffusion ?

In [3]:
# @title

platform_category_kpis = (
    campaign_kpis
    .groupby(
        ["platform", "campaign_category"],
        as_index=False
    )
    .agg(
        nb_campagnes=("campaign_id", "nunique"),
        inscriptions=("inscriptions", "sum"),
        budget=("cost", "sum"),
        impressions=("impressions", "sum")
    )
)

# Coût par inscription
platform_category_kpis["CPS"] = (
    platform_category_kpis["budget"]
    / platform_category_kpis["inscriptions"].replace(0, np.nan)
)


platform_category_kpis['budget_share'] = (
    platform_category_kpis['budget'] / platform_category_kpis['budget'] .sum() * 100
)
# Inscriptions attribuées par impression (%)
platform_category_kpis["Signup_Rate"] = (
    platform_category_kpis["inscriptions"]
    / platform_category_kpis["impressions"].replace(0, np.nan)
    * 100
)


platform_category_kpis["Signup_share"] = (
    platform_category_kpis["inscriptions"]
    / platform_category_kpis.groupby("platform")["inscriptions"].transform("sum")
    * 100
)



display(
    platform_category_kpis.sort_values(
        ["platform", "inscriptions"],
        ascending=[True, False]
    ).round(2)
)


,platform,campaign_category,nb_campagnes,inscriptions,budget,impressions,CPS,budget_share,Signup_Rate,Signup_share
1,Email,Health For All,1,3279,1185.17,88220,0.36,1.97,3.72,79.39
2,Email,Preventive Care News,6,643,6256.81,1116968,9.73,10.40,0.06,15.57
3,Email,Summer Wellness Tips,6,163,7085.68,573010,43.47,11.77,0.03,3.95
0,Email,Benefit Updates,2,45,2151.31,244280,47.81,3.57,0.02,1.09
5,SEO,Compare Health Coverage,8,2820,10043.21,664710,3.56,16.69,0.42,69.54
8,SEO,Tailored Health Plans,5,1107,5175.19,1398859,4.67,8.60,0.08,27.30
4,SEO,Affordable Plans,3,63,1569.15,327507,24.91,2.61,0.02,1.55
6,SEO,Family Coverage Plan,1,42,1606.23,449340,38.24,2.67,0.01,1.04
7,SEO,Golden Years Security,2,23,2853.22,348510,124.05,4.74,0.01,0.57
10,Social Media,#HealthyLiving,6,3727,6676.63,1372448,1.79,11.09,0.27,48.98


### Insight 2 — L'efficacité d'acquisition varie fortement selon les catégories au sein d'une même plateforme

>L'analyse croisée des plateformes et des catégories révèle une **forte hétérogénéité des performances d'acquisition**.
>
>Sur **Social Media**, les catégories #HealthyLiving et #CoverageMatters concentrent **95,4 % des inscriptions attribuées pour seulement 52,6 % du budget du canal**, avec des coûts par inscription respectifs de 1,79 \$ et 0,65 \$.
>
>Une concentration similaire apparaît sur **Email**, où Health For All représente **79,4 % des inscriptions du canal**, avec un coût par inscription de seulement 0,36 \$.
>
>Sur **SEO**, Compare Health Coverage domine également l'acquisition attribuée, avec 69,5 % des inscriptions de la plateforme.
>
>**Interprétation business :** les performances marketing ne sont pas homogènes au sein des plateformes. Les résultats globaux des canaux sont fortement influencés par certaines catégories et campagnes dominantes.
>
>**Implication stratégique :** l'optimisation du budget devrait tenir compte des combinaisons plateforme–catégorie, plutôt que de privilégier systématiquement une plateforme entière. Une analyse complémentaire sera nécessaire pour identifier les stratégies réellement reproductibles et vérifier la fiabilité de l'attribution marketing.

### .Q3) Quels types de campagnes fonctionnent le mieux selon les plateformes ?


In [4]:
# @title

campaign_type_kpis = (
    campaign_kpis
    .groupby(["platform", "campaign_type"])
    .agg(
        nb_campagnes=("campaign_id", "nunique"),
        inscriptions=("inscriptions", "sum"),
        budget=("cost", "sum"),
        impressions=("impressions", "sum")
    )
)

# Indicateurs d'efficacité
campaign_type_kpis["CPS"] = (
    campaign_type_kpis["budget"]
    / campaign_type_kpis["inscriptions"].replace(0, np.nan)
)

campaign_type_kpis["Signup_Rate"] = (
    campaign_type_kpis["inscriptions"]
    / campaign_type_kpis["impressions"].replace(0, np.nan)
    * 100
)

# Contributions internes à chaque plateforme
total_inscriptions = (
    campaign_type_kpis
    .groupby(level="platform")["inscriptions"]
    .transform("sum")
)

total_budget = (
    campaign_type_kpis
    .groupby(level="platform")["budget"]
    .transform("sum")
)

campaign_type_kpis["Signup_share"] = (
    campaign_type_kpis["inscriptions"]
    / total_inscriptions.replace(0, np.nan)
    * 100
)

campaign_type_kpis["Budget_share"] = (
    campaign_type_kpis["budget"]
    / total_budget.replace(0, np.nan)
    * 100
)

campaign_type_kpis["Allocation_gap"] = (
    campaign_type_kpis["Signup_share"]
    - campaign_type_kpis["Budget_share"]
)

campaign_type_kpis = campaign_type_kpis.sort_values(
    ["platform", "inscriptions"],
    ascending=[True, False]
)

display(campaign_type_kpis.round(2))


nb_campagnes  inscriptions   budget  \
platform     campaign_type                                               
Email        Health Awareness                 3          3281  3063.67   
             Customer Testimonial             1           586  1624.44   
             Policy Information               5           230  5113.23   
             Product Promotion                3            33  3347.11   
             Covid Awareness                  2             0  3171.41   
             Health Tips                      1             0   359.11   
SEO          Customer Testimonial             3          1588  4389.37   
             Product Promotion                2          1273  2654.45   
             Covid Awareness                  6          1017  6798.28   
             Health Awareness                 1           114   394.64   
             Offer Announcement               3            50  3942.51   
             Policy Information               2            12  1912.41   
             Health Tips                      2             1  1155.33   
Social Media Product Promotion                4          3781  2978.46   
             Policy Information               2          2745  2535.79   
             Health Tips                      3           971  3640.22   
             Customer Testimonial             2            55  2663.05   
             Covid Awareness                  2            47  3133.14   
             Offer Announcement               3            11  2138.91   
TV           Product Promotion                3           494  1985.79   
             Covid Awareness                  2             0   873.19   
             Health Awareness                 1             0  1062.21   
             Policy Information               1             0  1253.53   

                                   impressions      CPS  Signup_Rate  \
platform     campaign_type                                             
Email        Health Awareness           310540     0.93         1.06   
             Customer Testimonial       428100     2.77         0.14   
             Policy Information        1068155    22.23         0.02   
             Product Promotion          114289   101.43         0.03   
             Covid Awareness             58764      NaN         0.00   
             Health Tips                 42630      NaN         0.00   
SEO          Customer Testimonial       589766     2.76         0.27   
             Product Promotion          116890     2.09         1.09   
             Covid Awareness            914491     6.68         0.11   
             Health Awareness           480790     3.46         0.02   
             Offer Announcement         800350    78.85         0.01   
             Policy Information         134509   159.37         0.01   
             Health Tips                152130  1155.33         0.00   
Social Media Product Promotion          833213     0.79         0.45   
             Policy Information         593220     0.92         0.46   
             Health Tips                524341     3.75         0.19   
             Customer Testimonial       493810    48.42         0.01   
             Covid Awareness            480610    66.66         0.01   
             Offer Announcement         334118   194.45         0.00   
TV           Product Promotion          498432     4.02         0.10   
             Covid Awareness             60940      NaN         0.00   
             Health Awareness            29630      NaN         0.00   
             Policy Information          20437      NaN         0.00   

                                   Signup_share  Budget_share  Allocation_gap  
platform     campaign_type                                                     
Email        Health Awareness             79.44         18.37           61.07  
             Customer Testimonial         14.19          9.74            4.45  
             Policy Information            5.57         30.66  

### Insight 3 — L'efficacité des types de campagnes varie fortement selon leur plateforme de diffusion

L'analyse croisée des types de campagnes et des plateformes révèle d'importantes disparités dans les performances d'acquisition attribuée.

Le type **Product Promotion** illustre particulièrement ce phénomène : son coût par inscription atteint **0,79 USD sur Social Media**, contre **2,09 USD sur SEO, 4,02 USD sur TV et 101,43 USD sur Email**.

Sur **Social Media**, Product Promotion et Policy Information génèrent conjointement **85,75 % des inscriptions attribuées pour 32,27 % du budget du canal**.

Sur **Email**, Health Awareness se distingue en concentrant **79,44 % des inscriptions pour 18,37 % du budget**, avec un CPS de 0,93 USD. Cette performance reste toutefois fortement dépendante de CAM018, qui génère presque toutes les inscriptions de ce groupe.

**Interprétation business :** l'efficacité observée dépend de la combinaison entre le type de campagne et sa plateforme de diffusion. Cependant, ces écarts peuvent également être influencés par les caractéristiques et la concentration des campagnes individuelles.

**Implication stratégique :** les décisions d'optimisation budgétaire doivent être étudiées au niveau des combinaisons plateforme × type de campagne, puis vérifiées au niveau des campagnes individuelles, plutôt que de généraliser les performances d'un type ou d'un canal à l'ensemble du portefeuille.

### .Q4) Dans quelle mesure chaque plateforme marketing dépend-elle d'un petit nombre de campagnes pour générer ses inscriptions attribuées ?

In [5]:
# @title
resultats = campaign_kpis.groupby(["platform" , "campaign_category"]).agg(
    nb_campagnes=("campaign_id", "nunique"),
    inscriptions=("inscriptions", "sum"),
    budget=("cost", "sum"),
    impressions=("impressions", "sum"),
    clicks = ("clicks", "sum"))

resultats['clicks']=resultats['clicks'].replace(0 , np.nan)
resultats['impressions']=resultats['impressions'].replace(0 , np.nan)

resultats['CPS'] = resultats['budget'] / resultats['inscriptions'].replace(0, np.nan)
resultats['Signup_Rate'] = resultats['inscriptions'] / resultats['impressions'].replace(0, np.nan) * 100

resultats['Campaign_share'] = resultats['nb_campagnes'] / resultats.groupby('platform')['nb_campagnes'].transform('sum') * 100

resultats['Signup_share'] = resultats['inscriptions'] / resultats.groupby('platform')['inscriptions'].transform('sum') * 100

resultats['Budget_share'] = resultats['budget'] / resultats.groupby('platform')['budget'].transform('sum') * 100

resultats['Allocation_gap'] = resultats['Signup_share'] - resultats['Budget_share']

resultats['CTR'] = (
    resultats['clicks'] / resultats['impressions'] * 100
).replace(0, np.nan)

resultats = resultats.sort_values(["platform", "inscriptions"], ascending=[True, False])
# resultats = resultats.round(2)
display(resultats)

nb_campagnes  inscriptions  \
platform     campaign_category                                     
Email        Health For All                      1          3279   
             Preventive Care News                6           643   
             Summer Wellness Tips                6           163   
             Benefit Updates                     2            45   
SEO          Compare Health Coverage             8          2820   
             Tailored Health Plans               5          1107   
             Affordable Plans                    3            63   
             Family Coverage Plan                1            42   
             Golden Years Security               2            23   
Social Media #HealthyLiving                      6          3727   
             #CoverageMatters                    3          3536   
             #InsureYourHealth                   6           316   
             Family Coverage Plan                1            31   
TV           Health For All                      3           266   
             Family Coverage Plan                1           228   
             Golden Years Security               3             0   

                                            budget  impressions    clicks  \
platform     campaign_category                                              
Email        Health For All            1185.171031        88220   43453.0   
             Preventive Care News      6256.810157      1116968  136664.0   
             Summer Wellness Tips      7085.680129       573010  103668.0   
             Benefit Updates           2151.306272       244280   54145.0   
SEO          Compare Health Coverage  10043.207969       664710   93350.0   
             Tailored Health Plans     5175.185470      1398859   92570.0   
             Affordable Plans          1569.147489       327507   41594.0   
             Family Coverage Plan      1606.227668       449340       NaN   
             Golden Years Security     2853.224113       348510    5979.0   
Social Media #HealthyLiving            6676.631363      1372448  132068.0   
             #CoverageMatters          2311.331446       712100   74240.0   
             #InsureYourHealth         6572.881885       970244   74541.0   
             Family Coverage Plan      1528.728281       204520       NaN   
TV           Health For All            3161.795270        82339       NaN   
             Family Coverage Plan       801.456917       452680       NaN   
             Golden Years Security     1211.462688        74420       NaN   

                                             CPS  Signup_Rate  Campaign_share  \
platform     campaign_category                                                  
Email        Health For All             0.361443     3.716844        6.666667   
             Preventive Care News       9.730653     0.057567       40.000000   
             Summer Wellness Tips      43.470430     0.028446       40.000000   
             Benefit Updates           47.806806     0.018421       13.333333   
SEO          Compare Health Coverage    3.561421     0.424245       42.105263   
             Tailored Health Plans      4.674964     0.079136       26.315789   
             Affordable Plans          24.907103     0.019236       15.789474   
             Family Coverage Plan      38.243516     0.009347        5.263158   
             Golden Years Security    124.053222     0.006600       10.526316   
Social Media #HealthyLiving             1.791422     0.271559       37.500000   
             #CoverageMatters           0.653657     0.496559       18.750000   
             #InsureYourHealth         20.800259     0.032569       37.500000   
             Family Coverage Plan      49.313816     0.015157        6.250000   
TV           Health For All            11.886448     0.323055       42.857143   
             Family Coverage Plan       3.515162     0.050367       14.285714   
             Golden Years Security           NaN 

In [6]:
# @title

# 1. Classer les campagnes par volume d'inscriptions
concentration = campaign_kpis[
    ["platform", "campaign_id", "cost", "inscriptions"]
].copy()

concentration = concentration.sort_values(
    ["platform", "inscriptions"],
    ascending=[True, False]
)

concentration["rang"] = (
    concentration.groupby("platform").cumcount() + 1
)

# 2. Agrégation globale par plateforme
platform_concentration = concentration.groupby("platform").agg(
    Nb_Campagnes=("campaign_id", "nunique"),
    Total_Inscriptions=("inscriptions", "sum"),
    Budget_Total=("cost", "sum"),
    Nb_Zero_Signups=("inscriptions", lambda x: x.eq(0).sum())
)

# 3. Contributions du Top 1 et du Top 2
top1 = (
    concentration.query("rang == 1")
    .set_index("platform")
)

top2 = (
    concentration.query("rang <= 2")
    .groupby("platform")
    .agg(
        Top2_Inscriptions=("inscriptions", "sum"),
        Top2_Budget=("cost", "sum")
    )
)

platform_concentration["Top1_Campaign"] = top1["campaign_id"]
# platform_concentration["Top2_Campaign"] = top2.index
platform_concentration["Top1_Share"] = (
    top1["inscriptions"]
    / platform_concentration["Total_Inscriptions"]
    * 100
)

platform_concentration["Top2_Share"] = (
    top2["Top2_Inscriptions"]
    / platform_concentration["Total_Inscriptions"]
    * 100
)

platform_concentration["Top2_Budget_Share"] = (
    top2["Top2_Budget"]
    / platform_concentration["Budget_Total"]
    * 100
)

# 4. Ratio de concentration
platform_concentration["Concentration_Gap"] = (
    platform_concentration["Top2_Share"]
    - platform_concentration["Top2_Budget_Share"]
)

display(platform_concentration.round(2))


,Nb_Campagnes,Total_Inscriptions,Budget_Total,Nb_Zero_Signups,Top1_Campaign,Top1_Share,Top2_Share,Top2_Budget_Share,Concentration_Gap
platform,,,,,,,,,
Email,15,4130,16678.97,7,CAM018,79.39,93.58,16.85,76.74
SEO,19,4055,21246.99,7,CAM004,38.13,69.52,12.28,57.24
Social Media,16,7610,17089.57,5,CAM031,46.44,82.48,12.69,69.79
TV,7,494,5174.71,5,CAM033,53.85,100.00,31.84,68.16


### Insight 4 — Les inscriptions attribuées sont fortement concentrées sur deux campagnes par plateforme

L'analyse de concentration révèle une forte dépendance des plateformes à un nombre limité de campagnes.

Sur **Email**, les deux campagnes les plus performantes génèrent **93,58 % des inscriptions attribuées pour seulement 16,85 % du budget du canal**.

Cette concentration s'observe également sur **Social Media (82,48 % des inscriptions pour 12,69 % du budget)** et **SEO (69,52 % pour 12,28 %)**.

Sur **TV**, les deux meilleures campagnes génèrent la totalité des inscriptions attribuées, les cinq autres n'en comptabilisant aucune.

Au total, **24 campagnes sur 57 (42,11 %) ne présentent aucune inscription attribuée**.

**Interprétation business :** les performances d'acquisition des différentes plateformes reposent principalement sur un nombre restreint de campagnes. Cette concentration est particulièrement marquée sur Email et Social Media, tandis que SEO présente une répartition relativement moins concentrée.

**Implication stratégique :** les décisions budgétaires doivent intégrer le risque de dépendance à quelques campagnes dominantes. Les campagnes sans inscription attribuée nécessitent également une évaluation de leurs objectifs et de la fiabilité de leur attribution avant toute décision de réduction budgétaire.

> Les plateformes qui génèrent le plus de clics et d'engagement publicitaire sont-elles également celles qui enregistrent les meilleurs résultats d'acquisition attribuée ?

In [8]:

# @title Q5 — Digital Marketing Visibility KPIs

df = campaign_kpis.query("platform != 'TV'").copy()

# Campagnes avec données de clics exploitables
eligible = df["clicks"].notna() & df["impressions"].gt(0)

df["clicks_valid"] = df["clicks"].where(eligible)
df["impressions_valid"] = df["impressions"].where(eligible)
df["budget_valid"] = df["cost"].where(eligible)
df["has_click_data"] = eligible.astype(int)

visibility_kpis = df.groupby("platform").agg(
    nb_campagnes=("campaign_id", "nunique"),
    nb_campagnes_clicks=("has_click_data", "sum"),
    inscriptions=("inscriptions", "sum"),
    budget=("cost", "sum"),
    impressions=("impressions", "sum"),
    clicks=("clicks_valid", lambda x: x.sum(min_count=1)),
    impressions_valid=("impressions_valid", "sum"),
    budget_valid=("budget_valid", "sum")
)

# Acquisition
visibility_kpis["CPS"] = (
    visibility_kpis["budget"]
    / visibility_kpis["inscriptions"].replace(0, np.nan)
)

visibility_kpis["Signup_Rate"] = (
    visibility_kpis["inscriptions"]
    / visibility_kpis["impressions"].replace(0, np.nan)
    * 100
)

# Engagement - périmètre des clics renseignés
visibility_kpis["CPC"] = (
    visibility_kpis["budget_valid"]
    / visibility_kpis["clicks"].replace(0, np.nan)
)

visibility_kpis["CTR"] = (
    visibility_kpis["clicks"]
    / visibility_kpis["impressions_valid"].replace(0, np.nan)
    * 100
)

# Visibilité - toutes les campagnes numériques
visibility_kpis["CPM"] = (
    visibility_kpis["budget"]
    / visibility_kpis["impressions"].replace(0, np.nan)
    * 1000
)

# Parts relatives au marketing numérique uniquement
visibility_kpis["budget_share"] = (
    visibility_kpis["budget"]
    / visibility_kpis["budget"].sum()
    * 100
)

visibility_kpis["Signup_share"] = (
    visibility_kpis["inscriptions"]
    / visibility_kpis["inscriptions"].sum()
    * 100
)

visibility_kpis["Allocation_gap"] = (
    visibility_kpis["Signup_share"]
    - visibility_kpis["budget_share"]
)

# Contrôle de couverture
visibility_kpis["Click_Data_Coverage"] = (
    visibility_kpis["nb_campagnes_clicks"]
    / visibility_kpis["nb_campagnes"]
    * 100
)

visibility_kpis = visibility_kpis.sort_values(
    "CTR", ascending=False
)

display(visibility_kpis.round(3))


,nb_campagnes,nb_campagnes_clicks,inscriptions,budget,impressions,clicks,impressions_valid,budget_valid,CPS,Signup_Rate,CPC,CTR,CPM,budget_share,Signup_share,Allocation_gap,Click_Data_Coverage
platform,,,,,,,,,,,,,,,,,
Email,15,15,4130,16678.968,2022478,337930.0,2022478.0,16678.968,4.038,0.204,0.049,16.709,8.247,30.317,26.148,-4.169,100.000
Social Media,16,15,7610,17089.573,3259312,280849.0,3054792.0,15560.845,2.246,0.233,0.055,9.194,5.243,31.063,48.180,17.117,93.750
SEO,19,18,4055,21246.993,3188926,233493.0,2739586.0,19640.765,5.240,0.127,0.084,8.523,6.663,38.620,25.673,-12.947,94.737


### Insight 5 — Un taux de clics élevé ne garantit pas une meilleure efficacité d'acquisition

L'analyse des performances publicitaires révèle un décalage entre la génération de clics et les inscriptions attribuées.

**Email** enregistre le CTR le plus élevé (**16,71 %**) ainsi que le CPC le plus faible (**0,049 USD**), contre respectivement **9,19 %** et **0,055 USD** pour Social Media.

Cependant, **Social Media génère 84,3 % d'inscriptions attribuées supplémentaires**, avec un coût par inscription de **2,25 USD**, contre **4,04 USD pour Email**, soit un coût inférieur de 44,4 %.

Social Media présente également le CPM le plus faible parmi les canaux numériques, à **5,24 USD**, contre 8,25 USD pour Email.

**Interprétation business :** les performances publicitaires et les résultats d'acquisition attribuée ne suivent pas nécessairement la même hiérarchie. La capacité à générer des clics à faible coût ne garantit pas un meilleur coût par inscription.

**Implication stratégique :** l'évaluation des plateformes marketing doit intégrer conjointement les indicateurs de visibilité (CPM), de clics (CTR, CPC) et d'acquisition attribuée (CPS). Les décisions budgétaires devront également tenir compte des performances individuelles des campagnes et de la fiabilité des données d'attribution.

### .Q6) Quelle part du budget marketing est mobilisée par les campagnes sans aucune inscription attribuée ?

In [42]:

# @title Q6 — Budget des campagnes sans inscription attribuée

df = campaign_kpis.copy()
zero_signup = df.query("inscriptions == 0").copy()

# KPIs globaux
budget_total = df["cost"].sum()
budget_zero = zero_signup["cost"].sum()

pct_budget_zero = budget_zero / budget_total * 100
nb_zero = zero_signup["campaign_id"].nunique()

print(f"Campagnes sans inscription : {nb_zero}/{df['campaign_id'].nunique()}")
print(f"Budget sans inscription : {budget_zero:,.2f} USD")
print(f"Part du budget global : {pct_budget_zero:.2f}%")

# Performance par plateforme - toutes les campagnes
platform_total = df.groupby("platform").agg(
    nb_campagnes=("campaign_id", "nunique"),
    budget_total=("cost", "sum")
)

# Dépenses sans inscription, par plateforme
platform_zero = zero_signup.groupby("platform").agg(
    nb_sans_inscription=("campaign_id", "nunique"),
    budget_sans_inscription=("cost", "sum")
)

# Comparaison des périmètres
budget_analysis = platform_total.join(
    platform_zero,
    how="left"
).fillna(0)

# % du budget de la plateforme sans inscription attribuée
budget_analysis["pct_budget_platform"] = (
    budget_analysis["budget_sans_inscription"]
    / budget_analysis["budget_total"]
    * 100
)

# Contribution au budget marketing global
budget_analysis["pct_budget_global"] = (
    budget_analysis["budget_sans_inscription"]
    / budget_total
    * 100
)

# % de campagnes sans inscription dans chaque plateforme
budget_analysis["pct_campaigns_zero"] = (
    budget_analysis["nb_sans_inscription"]
    / budget_analysis["nb_campagnes"]
    * 100
)

display(budget_analysis.round(2))


Campagnes sans inscription : 24/57
Budget sans inscription : 24,709.05 USD
Part du budget global : 41.05%


,nb_campagnes,budget_total,nb_sans_inscription,budget_sans_inscription,pct_budget_platform,pct_budget_global,pct_campaigns_zero
platform,,,,,,,
Email,15,16678.97,7,7870.11,47.19,13.08,46.67
SEO,19,21246.99,7,8469.90,39.86,14.07,36.84
Social Media,16,17089.57,5,4841.84,28.33,8.04,31.25
TV,7,5174.71,5,3527.20,68.16,5.86,71.43


### Insight 6 — 41 % du budget marketing est associé à des campagnes sans inscription attribuée

>L'analyse des dépenses marketing révèle que **24 campagnes sur 57 (42,11 %) ne comptabilisent aucune inscription attribuée**, alors qu'elles mobilisent **24 709 USD, soit 41,05 % du budget total**.

>**SEO et Email concentrent 66,13 % de ces dépenses**, avec respectivement 8 470 USD et 7 870 USD engagés sur des campagnes sans inscription attribuée.
>
>En proportion du budget propre à chaque canal, **TV présente le ratio le plus élevé (68,16 %)**, devant Email (47,19 %), SEO (39,86 %) et Social Media (28,33 %).
>
>**Interprétation business :** une part importante des investissements marketing ne présente aucun résultat d'acquisition attribuée mesurable dans les données disponibles. Cette situation peut refléter des différences d'objectifs entre campagnes, des limites du système d'attribution ou des performances d'acquisition insuffisantes.
>
>**Implication stratégique :** une évaluation prioritaire des dépenses concernées sur SEO et Email est recommandée en raison de leur poids financier. Pour TV, l'analyse devra tenir compte des objectifs de notoriété et des indicateurs de performance appropriés. Toute décision de réduction ou de réallocation budgétaire devra être précédée d'une vérification de l'attribution et des objectifs des campagnes.

### .Q7) Quelles plateformes et quelles combinaisons de campagnes Row Health devrait-elle prioriser pour améliorer l'efficacité de ses dépenses marketing ?

In [43]:

# @title Q7 — Decision Matrix

df = campaign_kpis.copy()

# Identifier le budget sans inscription attribuée
df["budget_zero"] = df["cost"].where(
    df["inscriptions"].eq(0), 0
)

# Synthèse par plateforme
decision_kpis = df.groupby("platform").agg(
    nb_campagnes=("campaign_id", "nunique"),
    inscriptions=("inscriptions", "sum"),
    budget=("cost", "sum"),
    budget_zero=("budget_zero", "sum")
)

# Efficacité d'acquisition
decision_kpis["CPS"] = (
    decision_kpis["budget"]
    / decision_kpis["inscriptions"].replace(0, np.nan)
)

# Exposition financière
decision_kpis["Budget_Zero_pct"] = (
    decision_kpis["budget_zero"]
    / decision_kpis["budget"]
    * 100
)

# Concentration : 2 premières campagnes par plateforme
top2 = (
    df.sort_values(
        ["platform", "inscriptions"],
        ascending=[True, False]
    )
    .groupby("platform")
    .head(2)
    .groupby("platform")["inscriptions"]
    .sum()
)

decision_kpis["Top2_Share"] = (
    top2
    / decision_kpis["inscriptions"].replace(0, np.nan)
    * 100
)

# Priorité d'audit selon le montant financier
decision_kpis = decision_kpis.sort_values(
    "budget_zero", ascending=False
)

display(decision_kpis.round(2))


,nb_campagnes,inscriptions,budget,budget_zero,CPS,Budget_Zero_pct,Top2_Share
platform,,,,,,,
SEO,19,4055,21246.99,8469.90,5.24,39.86,69.52
Email,15,4130,16678.97,7870.11,4.04,47.19,93.58
Social Media,16,7610,17089.57,4841.84,2.25,28.33,82.48
TV,7,494,5174.71,3527.20,10.48,68.16,100.00


### Q7  Priorités stratégiques pour optimiser l'efficacité des dépenses marketing

>L'analyse croisée des performances d'acquisition, de l'exposition budgétaire et de la concentration des inscriptions révèle des opportunités d'optimisation distinctes selon les plateformes.
>
>**Social Media apparaît comme un candidat prometteur pour de futurs tests d'investissement**, avec le CPS le plus faible (2,25 USD) et le volume d'inscriptions attribuées le plus élevé (7 610). Toutefois, 82,48 % de ses inscriptions reposent sur deux campagnes, ce qui limite la possibilité de généraliser ses performances.
>
>À l'inverse, **SEO et Email constituent les principales priorités d'audit budgétaire**. Ces deux plateformes concentrent ensemble 16 340 USD de dépenses sans inscription attribuée, soit 66,13 % des dépenses concernées dans l'ensemble du portefeuille marketing.
>
>Enfin, **TV présente la proportion la plus élevée de budget sans inscription attribuée (68,16 %)**, mais son évaluation nécessite de considérer les objectifs de notoriété et les limites de l'attribution directe.
>
>**Interprétation business :** les écarts de performance entre plateformes ne justifient pas une réallocation uniforme du budget. Les opportunités et les risques se situent davantage au niveau des campagnes individuelles et de leurs objectifs.
>
>**Recommandations stratégiques :**
>
>1. **Auditer prioritairement SEO et Email** afin d'identifier les dépenses sans acquisition attribuée et d'en vérifier les objectifs et les mécanismes d'attribution.
>2. **Examiner les campagnes dominantes**, notamment CAM031, CAM030 et CAM018, avant d'envisager des investissements supplémentaires.
>3. **Tester progressivement des augmentations budgétaires ciblées**, en mesurant l'évolution du coût d'acquisition plutôt qu'en supposant que les performances historiques resteront constantes.
>4. **Adapter le cadre d'évaluation de TV** à ses objectifs marketing réels.
>
>**Conclusion :** l'optimisation des dépenses de Row Health doit privilégier une approche sélective, fondée sur les performances individuelles, la qualité de l'attribution et des expérimentations contrôlées, plutôt qu'une redistribution automatique du budget entre plateformes.